# 예제 01. 저축은행 경영지표 분석
### 금융데이터 분석의 첫걸음 · 1일 차 · 1시간

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`를 누르면 실행되고 다음 셀로 넘어갑니다.

| 항목 | 내용 |
|---|---|
| 데이터 | 예금보험공사 「저축은행업권 주요 경영지표 현황」 2026년 3월 말 · 저축은행 79개사 |
| 출처 | 공공데이터포털 data.go.kr/data/15120292 (원본 CSV, 이용허락범위 제한 없음) |
| 오늘의 결과물 | 저축은행 경영지표 비교표 + 막대그래프 |

**오늘 해결할 질문**: "국내 저축은행 79곳은 규모와 건전성이 어떻게 다를까?"

## 0. 실습 준비

아래 셀을 실행하면 실습 데이터를 **GitHub에서 자동으로 내려받습니다.** (강의장 인터넷이 막혀 **[파일 선택]** 버튼이 나오면, 강사가 배포한 파일을 고르면 됩니다.)
`데이터 준비 완료: True`가 나오면 성공입니다.

In [ ]:
import os
import urllib.request
import pandas as pd
import plotly.express as px

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "savings_bank.csv"
if not os.path.exists(FILE):          # 파일이 없으면 GitHub에서 내려받습니다
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
print("데이터 준비 완료:", os.path.exists(FILE))

## 01-1. CSV 파일 읽기

`pd.read_csv()`는 CSV 파일을 **표(DataFrame)** 로 불러옵니다. DataFrame은 파이썬 안의 **엑셀 시트**라고 생각하면 됩니다.
`df.head()`는 맨 위 5줄만 보여 줍니다.

In [ ]:
df = pd.read_csv(FILE)
df.head()

## 01-2. 몇 개 회사, 몇 개 항목인가?

`df.shape`는 (행 수, 열 수)입니다. 한 행이 저축은행 한 곳입니다.

In [ ]:
print("행(저축은행 수):", df.shape[0])
print("열(항목 수):", df.shape[1])
df.columns.tolist()

**항목 설명** (원본 컬럼명을 코드에서 쓰기 쉽게 줄였고, 값은 원본 그대로입니다)

| 컬럼 | 뜻 | 단위 |
|---|---|---|
| 저축은행명 · 지역 | 회사 이름, 본점 지역 | - |
| 총자산 | 회사가 가진 자산 전체 | 억원 |
| 기업대출 · 가계담보대출 · 가계신용대출 · 기타대출 | 대출금을 종류별로 나눈 것 | 억원 |
| 당기순이익 | 해당 기간 순이익 (마이너스는 적자) | 억원 |
| BIS비율 | BIS 기준 자기자본비율. **높을수록** 손실을 견딜 여력이 큼 | % |
| 연체율 | 대출 중 연체된 비율. **낮을수록** 좋음 | % |
| 고정이하여신비율 | 회수가 어려울 수 있는 부실 대출 비율. **낮을수록** 좋음 | % |
| 점포수 | 영업점 개수 | 개 |

## 01-3. 자료형 확인

숫자 항목이 숫자(`int64`, `float64`)로 들어왔는지 확인합니다. 글자 항목은 `object`(또는 `str`)로 표시됩니다.
숫자가 글자로 읽히면 합계·평균을 낼 수 없으니, 분석 전에 꼭 보는 단계입니다.

In [ ]:
df.info()

## 01-4. 빈 값(결측값) 확인

`isna().sum()`은 항목별로 비어 있는 칸의 개수를 셉니다. 모두 0이면 빈칸이 없는 깨끗한 데이터입니다.

In [ ]:
df.isna().sum()

## 01-5. 기초 통계 한눈에 보기

`describe()`는 숫자 항목의 개수·평균·최솟값·중앙값(50%)·최댓값을 한 번에 계산합니다.

In [ ]:
df.describe().round(1)

> 💡 **핵심 정리** — 총자산의 **평균은 약 1조 5,097억 원**인데 **중앙값(50%)은 5,812억 원**입니다.
> 대형사 몇 곳이 평균을 크게 끌어올렸다는 뜻입니다. 금융데이터에서는 평균만 보지 말고 중앙값을 같이 봅니다.

## 01-6. 대출금 합계 만들기

원본에는 대출이 종류별로만 있습니다. 네 종류를 더해 **대출금** 항목을 새로 만듭니다.
`df["새 항목"] = 계산식`처럼 쓰면 새 열이 생깁니다.

In [ ]:
대출항목 = ["기업대출", "가계담보대출", "가계신용대출", "기타대출"]
df["대출금"] = df[대출항목].sum(axis=1)       # axis=1 : 가로(한 회사 안에서) 더하기
df[["저축은행명", "총자산"] + 대출항목 + ["대출금"]].head()

## 01-7. 업계 전체 규모

79개사를 모두 더하면 저축은행 업계 전체 규모가 나옵니다.

In [ ]:
전체자산 = df["총자산"].sum()
상위10자산 = df.sort_values("총자산", ascending=False).head(10)["총자산"].sum()

print("79개사 총자산 합계:", 전체자산, "억원")
print("상위 10개사 비중:", round(상위10자산 / 전체자산 * 100, 1), "%")
print("적자(당기순이익 < 0) 회사 수:", (df["당기순이익"] < 0).sum(), "곳")

## 01-8. 지역별 저축은행 수

`value_counts()`는 값마다 몇 번 나오는지 셉니다.

In [ ]:
df["지역"].value_counts()

## 01-9. 총자산 상위 10개 저축은행 비교표 ⭐ 결과물 ①

`sort_values("기준", ascending=False)`로 큰 순서대로 정렬하고 `head(10)`으로 위의 10곳만 고릅니다.

In [ ]:
보는항목 = ["저축은행명", "지역", "총자산", "대출금", "당기순이익", "BIS비율", "연체율", "고정이하여신비율"]

top10 = df.sort_values("총자산", ascending=False).head(10)[보는항목]
top10

## 01-10. 막대그래프로 그리기 ⭐ 결과물 ②

`px.bar()`에 표, x축 항목, y축 항목만 알려 주면 그래프가 됩니다. 막대에 마우스를 올리면 값이 보입니다.

In [ ]:
fig = px.bar(top10, x="저축은행명", y="총자산", text="총자산",
             title="총자산 상위 10개 저축은행 (2026년 3월 말, 억원)")
fig.show()

## 01-11. 건전성 지표 함께 보기

y에 항목을 **여러 개** 주고 `barmode="group"`을 쓰면 막대를 나란히 비교할 수 있습니다.
점선은 79개사 **연체율 중앙값**입니다.

In [ ]:
fig = px.bar(top10, x="저축은행명", y=["연체율", "고정이하여신비율"], barmode="group",
             title="총자산 상위 10개사의 연체율 · 고정이하여신비율 (%)")
fig.add_hline(y=df["연체율"].median(), line_dash="dot",
              annotation_text="79개사 연체율 중앙값")
fig.show()

## 01-12. 🔧 조건 바꿔 보기 (실습)

`기준`과 `개수`만 바꾸고 다시 실행해 보세요.

- `기준`: `"총자산"`, `"대출금"`, `"당기순이익"`, `"BIS비율"`, `"연체율"`, `"점포수"`
- `낮은순 = True`로 바꾸면 **작은 값부터** 보여 줍니다 (예: 연체율이 낮은 곳)

In [ ]:
기준 = "BIS비율"      # ← 바꿔 보세요
개수 = 10
낮은순 = False

결과 = df.sort_values(기준, ascending=낮은순).head(개수)
fig = px.bar(결과, x="저축은행명", y=기준, text=기준, title=f"{기준} 기준 {개수}개사")
fig.show()

## 01-13. [선택] 우리 지역 저축은행만 보기

`df[조건]`처럼 쓰면 조건에 맞는 행만 골라냅니다. `지역`을 `"부산"`, `"대구"` 등으로 바꿔 보세요.

In [ ]:
지역 = "광주"
df[df["지역"] == 지역][보는항목]

## 01-14. [선택] 규모와 연체율의 관계 (산점도)

점 하나가 저축은행 한 곳입니다. 점에 마우스를 올리면 회사 이름이 보입니다.
총자산은 회사마다 차이가 너무 커서 `log_x=True`(로그 눈금)로 고르게 펼쳤습니다.

In [ ]:
fig = px.scatter(df, x="총자산", y="연체율", color="지역", hover_name="저축은행명",
                 log_x=True, title="총자산(로그 눈금)과 연체율")
fig.show()

## 01-15. 해석할 때 꼭 기억할 점 ⚠️

- **지표 순위만으로 특정 회사의 경영 위험을 단정하지 않습니다.** 연체율이 높아도 충당금·자본 여력·대출 구성에 따라 상황이 다릅니다.
- 이 데이터는 **한 시점(2026년 3월 말)** 의 모습입니다. 추세는 여러 분기 데이터를 이어 봐야 압니다 (공공데이터포털에 과거 분기 자료가 있습니다).
- 저축은행이 금융감독원에 보고한 업무보고서 기준이며, 이후 감사·검사 결과에 따라 **수정될 수 있습니다.**
- 총자산이 매우 작은 회사는 비율 지표가 크게 흔들릴 수 있습니다 (표 맨 아래 회사들의 수치를 확인해 보세요).

**💬 토의**: 우리 회사 지표를 업계 중앙값과 비교한다면 어떤 항목을 가장 먼저 보겠습니까?

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| CSV 읽기 | `pd.read_csv("파일")` |
| 크기·자료형 확인 | `df.shape`, `df.info()` |
| 빈 값 확인 | `df.isna().sum()` |
| 기초 통계 | `df.describe()` |
| 새 항목 만들기 | `df["새항목"] = 계산식` |
| 정렬해서 상위 N개 | `df.sort_values("기준", ascending=False).head(N)` |
| 막대그래프 | `px.bar(표, x="항목", y="항목")` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 당기순이익 상위 5개 저축은행의 이름과 당기순이익을 표로 보여 주세요.
**이렇게 나오면 성공!** 한국투자 980 → OK 820 → 웰컴 452 → SBI 154 → 상상인플러스 139

**미션 2.** 연체율이 **가장 낮은** 5개 저축은행을 보여 주세요. (힌트: `ascending=True`)
**이렇게 나오면 성공!** 대원 0.0 → 대아 0.8 → 오성 1.8 → 센트럴 2.1 → 유안타 2.7
(1위·2위는 총자산 100억 원 미만의 아주 작은 회사입니다. 01-15의 주의점을 떠올려 보세요.)

**다음 예제 전에 이것만은 꼭!**
- [ ] `head()`, `info()`, `describe()`로 데이터를 먼저 살펴보는 습관
- [ ] `sort_values()` + `head()`로 순위 표 만들기

In [ ]:
# 미션 1
df.sort_values("총자산", ascending=False).head(5)[["저축은행명", "총자산"]]   # ← "총자산" 두 곳을 "당기순이익"으로

In [ ]:
# 미션 2
df.sort_values("연체율", ascending=False).head(5)[["저축은행명", "총자산", "연체율"]]   # ← False를 True로

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
df.sort_values("당기순이익", ascending=False).head(5)[["저축은행명", "당기순이익"]]

In [ ]:
# 미션 2 정답
df.sort_values("연체율", ascending=True).head(5)[["저축은행명", "총자산", "연체율"]]

---
### ▶ 다음 예제 02. 고객 데이터 분석과 예금 가입 패턴 탐색
회사 단위 지표에서 **고객 단위**로 내려갑니다. 4만 5천여 명의 고객 기록에서 "어떤 고객이 정기예금에 가입했을까?"를 찾아봅니다.